# 심화 프로젝트: 대규모 시계열 데이터 최적화 및 이상 행위 탐지

## 배경 시나리오

당신은 대형 커머스 기업의 데이터 엔지니어입니다. 서버에는 매일 수천만 건의 사용자 행동 로그가 쌓입니다. 로그 데이터(df)는 다음과 같은 구조를 가지고 있습니다.

| 컬럼 | 설명 |
|------|------|
| `user_id` | 사용자 고유 번호 (정수형) |
| `timestamp` | 행동 발생 시간 (Datetime 객체) |
| `action_type` | 수행한 행동 (`'view'`, `'click'`, `'purchase'` 등) |
| `amount` | 결제 시 금액 (결제가 아닐 경우 NaN) |

## 과제 요구사항

다음 세 가지 기능을 수행하는 함수 `analyze_user_behavior(df)`를 구현하세요.

> **조건:** 1,000만 행 이상의 데이터를 처리해야 하므로 `for loop`나 `iterrows()` 사용은 엄격히 금지하며, 반드시 벡터화 연산(Vectorization)만 사용해야 합니다.

### 1. 세션 간격 계산
각 사용자별로 직전 활동으로부터 현재 활동까지 걸린 시간(초 단위)을 계산하여 `time_delta` 컬럼을 생성하세요. 첫 활동의 경우 0으로 처리합니다.

### 2. 광속 클릭커 탐지
동일한 사용자가 1초 미만의 간격으로 연속해서 5번 이상의 행동을 보인 경우, 해당 행들에 `is_bot` 플래그를 `True`로 표시하세요. Rolling 윈도우 활용을 권장합니다.

### 3. 메모리 최적화
분석 완료 후, 수치형 데이터(`int`, `float`)들을 데이터 손실이 없는 선에서 가장 작은 bit의 타입(예: `int64` → `int32`, `float64` → `float32`)으로 자동 변환하여 반환하세요.

In [5]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1_000
base_time = pd.Timestamp("2026-05-03")

bot_case = [base_time + pd.Timedelta(seconds=0.3 * i) for i in range(6)]
df_bot = pd.DataFrame({
    "user_id": 1001,
    "timestamp": bot_case,
    "action_type": "click",
    "amount": np.nan
})

normal_case = [base_time + pd.Timedelta(seconds=30 * i) for i in range(5)]
df_normal = pd.DataFrame({
    "user_id": 1002,
    "timestamp": normal_case,
    "action_type": "view",
    "amount": np.nan
})

df = pd.concat([df_bot, df_normal], ignore_index=True)
df


,user_id,timestamp,action_type,amount
0,1001,2026-05-03 00:00:00.000000000,click,NaN
1,1001,2026-05-03 00:00:00.300000000,click,NaN
2,1001,2026-05-03 00:00:00.600000000,click,NaN
3,1001,2026-05-03 00:00:00.899999999,click,NaN
4,1001,2026-05-03 00:00:01.200000000,click,NaN
5,1001,2026-05-03 00:00:01.500000000,click,NaN
6,1002,2026-05-03 00:00:00.000000000,view,NaN
7,1002,2026-05-03 00:00:30.000000000,view,NaN
8,1002,2026-05-03 00:01:00.000000000,view,NaN
9,1002,2026-05-03 00:01:30.000000000,view,NaN


In [13]:
def analyze_user_behavior(df):
    #1.세션 간격 계산
    df_id_time = df.sort_values(['user_id', 'timestamp']).copy()
    # print(f"{df_id_time=}")
    df['time_delta'] = (
        df_id_time.groupby('user_id')['timestamp']
        .diff()
        .dt.total_seconds()
        .fillna(0)
        )

    #2.광속 클릭커 탐지
    is_fast = (df['time_delta']<1) & (df['time_delta']>0)
    fast_int = is_fast.astype(np.int8)
    # print(f"{fast_int=}, {type(fast_int)=}")
    fast_count = (
        fast_int
        .groupby(df['user_id'])
        .transform(
            lambda x : x.rolling(window=5).sum()
        )
    )
    df['is_bot'] = fast_count >= 5

    #3.메모리 최적화
    int_cols = df.select_dtypes('int').columns
    float_cols = df.select_dtypes('float').columns
    df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast="integer")
    df[float_cols] = df[float_cols].apply(pd.to_numeric, downcast="integer")

    return df

result = analyze_user_behavior(df)
print(result)


    user_id                     timestamp action_type  amount  time_delta  \
0      1001 2026-05-03 00:00:00.000000000       click     NaN         0.0   
1      1001 2026-05-03 00:00:00.300000000       click     NaN         0.3   
2      1001 2026-05-03 00:00:00.600000000       click     NaN         0.3   
3      1001 2026-05-03 00:00:00.899999999       click     NaN         0.3   
4      1001 2026-05-03 00:00:01.200000000       click     NaN         0.3   
5      1001 2026-05-03 00:00:01.500000000       click     NaN         0.3   
6      1002 2026-05-03 00:00:00.000000000        view     NaN         0.0   
7      1002 2026-05-03 00:00:30.000000000        view     NaN        30.0   
8      1002 2026-05-03 00:01:00.000000000        view     NaN        30.0   
9      1002 2026-05-03 00:01:30.000000000        view     NaN        30.0   
10     1002 2026-05-03 00:02:00.000000000        view     NaN        30.0   

    is_bot  
0    False  
1    False  
2    False  
3    False  
4    False